# Interactive time-series geospatial data on a web map <img align="right" src="../Supplementary_data/dea_logo.jpg">

* [**Sign up to the DEA Sandbox**](https://docs.dea.ga.gov.au/setup/sandbox.html) to run this notebook interactively from a browser
* **Compatibility:** Notebook currently compatible with the `DEA Sandbox` environment
* **Products used:** 
[ga_ls_fc_3](https://explorer.sandbox.dea.ga.gov.au/products/ga_ls_fc_3)

## Background

[Leafmap](https://leafmap.org) is a Python package that bridges powerful geospatial analysis libraries with interactive web mapping. Built on top of [Leaflet](https://leafletjs.com/), it allows users to create and manipulate dynamic maps directly within a Jupyter notebook or JupyterLab environment, without needing to write JavaScript.

Visualising a satellite image time series on an interactive map enables exploration of spatial and temporal patterns that can be difficult to interpret from static plots alone. By interactively scrolling through time, zooming, and toggling layers, users can more intuitively assess changes in land cover, vegetation, or surface conditions across space and time.

## Description

This notebook demonstrates how to visualise a satellite time series data on an interactive web map.

---

## Getting started
To run this analysis, run all the cells in the notebook.

In [1]:
# Install the leafmap module. Afterwards, restart the `Kernel` from the top menu before continuing
# !pip install -q leafmap

### Load packages

In [2]:
import os
import odc.ui
import leafmap
import datacube
import odc.geo.xr
import numpy as np
import pandas as pd
from ipyleaflet import ImageOverlay
from ipywidgets import widgets as w
from IPython.display import display

### Connect to the datacube

In [3]:
dc = datacube.Datacube(app='Imagery_on_web_map')

## Analysis Parameters

In [4]:
# Set the central latitude and longitude
central_lat,central_lon = -35.9877, 145.6788

# Set the buffer to load around the central coordinates. 
buffer = 0.1

#time range to load
time = ("2024")

# Compute the bounding box for the study area
latitude = (central_lat - buffer, central_lat + buffer)
longitude = (central_lon - buffer, central_lon + buffer)

## Define a function to parameterise the web mapping feature

In [5]:
def da_to_timeseries_imageoverlay(data, opacity=1):
    """
    Converts an xarray.DataArray into a dictionary with
    times as keys and ipyleaflet ImageOverlays as values.

    Parameters
    ----------
    da : xarray dataarray or a numpy ndarray

    Returns
    -------
    out : dictionary
    bounds: bounds of input da

    """
    out = {}
    for time in data.time.values:
        date = str(pd.to_datetime(time).date())
        print("\r", "Opening:", date, end="")
        single = data.sel(time=time)
        vmin, vmax = np.nanpercentile(data.data, [2, 98])
        rgba = single.odc.colorize(vmin=vmin, vmax=vmax)
        data_url = rgba.odc.compress(as_data_url=True)
        (x1, y1), _, (x2, y2) = rgba.odc.geobox.extent.exterior.to_crs(
            "epsg:4326"
        ).points[:3]
        bounds = [[y1, x1], [y2, x2]]
        raster = ImageOverlay(url=data_url, bounds=bounds, opacity=opacity)
        out[date] = raster
    return out, bounds

## Load DEA Fractional Cover data

In [6]:
# Load the fractional cover of green vegetation from the DEA Fractional Cover product
fc = dc.load(product='ga_ls_fc_3',
             measurements=['pv'],
             output_crs='EPSG:3577',
             resolution=(-30, 30),
             group_by='solar_day',
             x=longitude,
             y=latitude,
             time = time,
             progress_cbk=odc.ui.with_ui_cbk(),
)

## Resample the time series

We will convert the fractional cover time series to a monthly cadence so we limit noise from clouds etc., and reduce the number of time-steps we are plotting

In [7]:
fc = fc.resample(time='MS').mean()

## Create and set up leafmap

In [8]:
# Use the parameterising function to set up the web mapping variables
time_dict, bounds = da_to_timeseries_imageoverlay(fc["pv"], opacity=1)

# Create a centered leafmap, use centriod to find centre of map
x, y = fc.odc.geobox.geographic_extent.centroid.coords[0]

#create leafmap object
m = leafmap.Map(center = [y,x],
               zoom=11)

# Add the dictionary of images to view on the time slider
m.add_time_slider(time_dict)  

 Opening: 2024-12-01

## Display the interactive map. 
Press `play` on the map to run the time-series.
The percentage of photosynthetic vegetation from the DEA Fractional Cover product is shown with yellow representing low values, green showing mid values and blue representing high values.

As we have plotted the entire year of 2024, we can see the cropping schedules for this region in Victoria.

In [9]:
display(m)

Map(center=[-35.987816341032435, 145.67901648184505], controls=(ZoomControl(options=['position', 'zoom_in_text…

---

## Additional information

**License:** The code in this notebook is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0). 
Digital Earth Australia data is licensed under the [Creative Commons by Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license.

**Contact:** If you need assistance, please post a question on the [Open Data Cube Discord chat](https://discord.com/invite/4hhBQVas5U) or on the [GIS Stack Exchange](https://gis.stackexchange.com/questions/ask?tags=open-data-cube) using the `open-data-cube` tag (you can view previously asked questions [here](https://gis.stackexchange.com/questions/tagged/open-data-cube)).
If you would like to report an issue with this notebook, you can file one on [GitHub](https://github.com/GeoscienceAustralia/dea-notebooks).

**Last modified:** October 2025

**Compatible datacube version:** 

In [10]:
print(datacube.__version__)

1.9.9


## Tags
Browse all available tags on the DEA User Guide's [Tags Index](https://docs.dea.ga.gov.au/genindex.html)